# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [6]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (17609, 20)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [7]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5362,19074106.39,3812.53,1228.37,27435
Audio,4079,9243632.73,2434.46,1096.16,15295
Mobilier,4090,8497117.66,2232.56,773.43,18569
Accessoire,4002,8479325.42,2290.47,880.76,15618


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [8]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   481.98   19074106.39               0.0025
1      Mobilier   140.96    8497117.66               0.0017
2         Audio  2073.36    9243632.73               0.0224
3  Informatique  2281.00   19074106.39               0.0120
4      Mobilier  2116.10    8497117.66               0.0249

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [9]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,525,P0004,0.166
telephone,237,P0025,0.160
web,650,P0004,0.177
BOUTIQUE,724,P0024,0.182
TELEPHONE,364,P0015,0.168
WEB,1060,P0005,0.171
boutique,4684,P0018,0.161
telephone,2352,P0036,0.168
web,7013,P0014,0.171


### Partie autonome

In [10]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

# TODO
for nom in ['sales', 'sales_enriched', 'ventes', 'raw_sales', 'df_sales', 'orders']:
    if nom in globals() and isinstance(globals()[nom], pd.DataFrame):
        df = globals()[nom].copy()
        print('Ventes trouvées dans :', nom)
        break
else:
    raise NameError("Aucun DataFrame de ventes trouvé : exécutez les cellules du haut (Run Above)")

Ventes trouvées dans : sales


In [19]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

# TODO
d = df.copy()
d['date_commande'] = pd.to_datetime(d['date_commande'].astype('string').str.strip(),
                                    format='mixed', dayfirst=True, errors='coerce')

clients_stats = d.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
)

# Panier moyen = CA / nombre de commandes (et non moyenne des lignes,
# une commande pouvant contenir plusieurs lignes)
clients_stats['panier_moyen'] = (clients_stats['ca_total']
                                 / clients_stats['nb_commandes']).round(2)
clients_stats['ca_total'] = clients_stats['ca_total'].round(2)

print('Nombre de clients :', len(clients_stats))
print()
print('Top 10 des clients par CA :')
top10 = clients_stats.nlargest(10, 'ca_total')
top10

Nombre de clients : 3052

Top 10 des clients par CA :


,nb_commandes,ca_total,premiere_commande,derniere_commande,nb_categories,panier_moyen
id_client,,,,,,
C00696,7,1110587.13,NaT,NaT,3,158655.30
C00279,10,1094384.59,2023-10-21 08:30:00,2023-10-21 08:30:00,4,109438.46
C02327,4,955731.99,2024-12-28 14:36:00,2024-12-28 14:36:00,2,238933.00
C02631,10,927252.63,2024-03-03 14:23:00,2024-03-03 14:23:00,4,92725.26
C01626,9,886033.34,2023-01-16 13:14:00,2023-12-22 15:36:00,4,98448.15
C02285,7,859509.11,2023-10-01 10:43:00,2023-10-01 10:43:00,2,122787.02
C00355,13,819941.13,2023-02-12 11:09:00,2024-08-20 15:08:00,4,63072.39
C00806,5,785773.49,NaT,NaT,2,157154.70
C00941,7,783206.51,NaT,NaT,3,111886.64


In [20]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

# TODO
mediane_cat = d.groupby('categorie')['montant'].transform('median')
d['montant_eleve'] = d['montant'] > 1.5 * mediane_cat

part = d['montant_eleve'].mean()
print(f"Commandes > 150 % de la médiane de leur catégorie : "
      f"{int(d['montant_eleve'].sum())} sur {len(d)} ({part:.1%})")
print()
print('Part par catégorie :')
print(d.groupby('categorie')['montant_eleve'].mean().mul(100).round(1)
      .sort_values(ascending=False).astype(str) + ' %')

Commandes > 150 % de la médiane de leur catégorie : 5519 sur 17609 (31.3%)

Part par catégorie :
categorie
Audio           34.1 %
Informatique    33.3 %
Mobilier        31.9 %
Accessoire      26.0 %
Name: montant_eleve, dtype: str


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [13]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 17609
Après jointure produits : 17609
Lignes sans produit correspondant : 76 (0.43%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [14]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
17609 lignes -> 17915 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [15]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [21]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

# TODO
def joindre(left, right, cle, nom):
    """Left join contrôlé : validate, contrôle du nombre de lignes, orphelins signalés."""
    left, right = left.copy(), right.copy()
    left[cle] = left[cle].astype(str).str.strip()
    right[cle] = right[cle].astype(str).str.strip()

    n_avant = len(left)
    out = left.merge(right, on=cle, how='left', validate='many_to_one',
                     suffixes=('', f'_{nom}'), indicator='_statut')

    assert len(out) == n_avant, f'{nom} : {n_avant} -> {len(out)} lignes, jointure qui duplique !'

    orphelins = out['_statut'].eq('left_only')
    out[f'{nom}_inconnu'] = orphelins
    out = out.drop(columns='_statut')
    print(f'{nom:<10} {n_avant} -> {len(out)} lignes | clés orphelines : '
          f'{int(orphelins.sum())} ({orphelins.mean():.2%})')
    return out


analytique = joindre(sales, products, 'id_produit', 'produit')
analytique = joindre(analytique, customers, 'id_client', 'client')
analytique = joindre(analytique, stores, 'id_magasin', 'magasin')

print('Forme finale :', analytique.shape)

produit    17609 -> 17609 lignes | clés orphelines : 76 (0.43%)
client     17609 -> 17609 lignes | clés orphelines : 0 (0.00%)
magasin    17609 -> 17609 lignes | clés orphelines : 0 (0.00%)
Forme finale : (17609, 37)


In [29]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

    # TODO
def safe_merge(left, right, **kwargs):
    kwargs = dict(kwargs)
    kwargs.pop('indicator', None)                 # on gère l'indicateur nous-mêmes
    how = kwargs.get('how', 'inner')

    merged = left.merge(right, indicator='_merge_status', **kwargs)

    # Taux de clés non appariées côté gauche
    n_non_appariees = int((merged['_merge_status'] == 'left_only').sum())
    taux = n_non_appariees / len(left) if len(left) else 0.0
    print(f'safe_merge : {len(left)} -> {len(merged)} lignes | '
          f'clés non appariées : {n_non_appariees} ({taux:.2%})')

    if how == 'left' and len(merged) != len(left):
        raise ValueError(
            f'Left join : {len(left)} lignes attendues, {len(merged)} obtenues. '
            'La table de droite contient probablement des clés en double.'
        )

    return merged.drop(columns='_merge_status')


# Test 1 : jointure normale
_ = safe_merge(sales, products, on='id_produit', how='left')

# Test 2 : référentiel avec un doublon -> doit lever l'exception
produits_doublon = pd.concat([products, products.head(1)])
try:
    safe_merge(sales, produits_doublon, on='id_produit', how='left')
except ValueError as e:
    print('Exception attendue :', e)

safe_merge : 17609 -> 17609 lignes | clés non appariées : 76 (0.43%)
safe_merge : 17609 -> 18075 lignes | clés non appariées : 76 (0.43%)
Exception attendue : Left join : 17609 lignes attendues, 18075 obtenues. La table de droite contient probablement des clés en double.


---
## Atelier 6.3 — Formats long et large (35 min)

In [28]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{int(month):02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,77902.0,73677.0,86989.0,69390.0,57547.0,58872.0,69444.0,38069.0,215475.0,78125.0,79700.0,106665.0
Audio,78481.0,85619.0,77190.0,818801.0,76763.0,81485.0,221894.0,74287.0,93501.0,89630.0,740597.0,108342.0
Informatique,1079812.0,785938.0,247514.0,106698.0,99229.0,143119.0,124853.0,494057.0,110438.0,133315.0,157100.0,1103227.0
Mobilier,72571.0,66684.0,58882.0,66268.0,549300.0,56594.0,71259.0,41391.0,676621.0,51873.0,82772.0,58279.0


In [24]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,1.0,77902.0
1,Audio,1.0,78481.0
2,Informatique,1.0,1079812.0
3,Mobilier,1.0,72571.0
4,Accessoire,2.0,73677.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [32]:
# On repart du format large actuel pour être sûr que long est à jour
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')

# Long -> large sans pivot_table
wide_back = long.set_index(['categorie', 'mois'])['ca'].unstack('mois')
wide_back = wide_back.reindex(index=wide.index, columns=wide.columns)

pd.testing.assert_frame_equal(wide_back, wide, check_names=False, check_dtype=False)
print('OK : on retrouve exactement le tableau de départ', wide_back.shape)
wide_back.head()

OK : on retrouve exactement le tableau de départ (4, 12)


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,77902.0,73677.0,86989.0,69390.0,57547.0,58872.0,69444.0,38069.0,215475.0,78125.0,79700.0,106665.0
Audio,78481.0,85619.0,77190.0,818801.0,76763.0,81485.0,221894.0,74287.0,93501.0,89630.0,740597.0,108342.0
Informatique,1079812.0,785938.0,247514.0,106698.0,99229.0,143119.0,124853.0,494057.0,110438.0,133315.0,157100.0,1103227.0
Mobilier,72571.0,66684.0,58882.0,66268.0,549300.0,56594.0,71259.0,41391.0,676621.0,51873.0,82772.0,58279.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [33]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [34]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

# TODO
from pathlib import Path

star_dir = (PROCESSED_DIR if 'PROCESSED_DIR' in globals()
            else Path.cwd().parent / 'data' / 'processed') / 'star'
star_dir.mkdir(parents=True, exist_ok=True)

# ---------- Table de faits ----------
f = sales.copy()
f['date_commande'] = pd.to_datetime(f['date_commande'], errors='coerce')
for col in ['quantite', 'prix_unitaire', 'remise_pct']:
    f[col] = pd.to_numeric(f[col], errors='coerce')

remise = f['remise_pct'].fillna(0)
if remise.max() > 1:                       # remise en % (ex. 10) -> fraction
    remise = remise / 100
f['montant'] = f['quantite'] * f['prix_unitaire']
f['montant_net'] = f['montant'] * (1 - remise)

# Une vente sans date ne peut pas être rattachée à dim_date : on l'écarte (documenté)
n_sans_date = int(f['date_commande'].isna().sum())
f = f.dropna(subset=['date_commande'])
f['cle_date'] = f['date_commande'].dt.strftime('%Y%m%d').astype(int)

for cle in ['id_client', 'id_produit', 'id_magasin']:
    f[cle] = f[cle].astype(str).str.strip()

# ---------- Dimensions dédoublonnées sur leur clé primaire ----------
def build_dim(df, cle):
    dim = df.copy()
    dim[cle] = dim[cle].astype(str).str.strip()
    avant = len(dim)
    dim = dim.drop_duplicates(subset=cle, keep='first')
    print(f'{cle:<12} {avant} -> {len(dim)} lignes après dédoublonnage')
    # Membre "INCONNU" pour accueillir les clés orphelines de la table de faits
    inconnu = pd.DataFrame({cle: ['INCONNU']})
    return pd.concat([dim, inconnu], ignore_index=True)

dim_customer = build_dim(customers, 'id_client')
dim_product = build_dim(products, 'id_produit')
dim_store = build_dim(stores, 'id_magasin')

# Clés orphelines -> 'INCONNU' (on garde la vente, on signale le problème)
for cle, dim in [('id_client', dim_customer), ('id_produit', dim_product),
                 ('id_magasin', dim_store)]:
    orphelines = ~f[cle].isin(dim[cle])
    print(f'{cle:<12} clés orphelines remplacées par INCONNU : {int(orphelines.sum())}')
    f.loc[orphelines, cle] = 'INCONNU'

fact_sales = f[['cle_date', 'id_client', 'id_produit', 'id_magasin', 'quantite',
                'prix_unitaire', 'remise_pct', 'montant', 'montant_net', 'statut']]
print(f'Ventes écartées faute de date : {n_sans_date}')

# ---------- Vérification d'intégrité ----------
assert fact_sales['cle_date'].isin(dim_date['cle_date']).all(), 'cle_date orpheline'
assert fact_sales['id_client'].isin(dim_customer['id_client']).all()
assert fact_sales['id_produit'].isin(dim_product['id_produit']).all()
assert fact_sales['id_magasin'].isin(dim_store['id_magasin']).all()
print('Intégrité référentielle : OK')

# ---------- Export Parquet ----------
for nom, table in {'fact_sales': fact_sales, 'dim_date': dim_date,
                   'dim_customer': dim_customer, 'dim_product': dim_product,
                   'dim_store': dim_store}.items():
    table.to_parquet(star_dir / f'{nom}.parquet', index=False)
    print('écrit :', f'{nom}.parquet', table.shape)

id_client    3060 -> 3060 lignes après dédoublonnage
id_produit   39 -> 39 lignes après dédoublonnage
id_magasin   9 -> 9 lignes après dédoublonnage
id_client    clés orphelines remplacées par INCONNU : 0
id_produit   clés orphelines remplacées par INCONNU : 13
id_magasin   clés orphelines remplacées par INCONNU : 0
Ventes écartées faute de date : 14102
Intégrité référentielle : OK
écrit : fact_sales.parquet (3507, 10)
écrit : dim_date.parquet (730, 10)
écrit : dim_customer.parquet (3061, 5)
écrit : dim_product.parquet (40, 5)
écrit : dim_store.parquet (10, 5)


In [35]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""
    problemes = {}
    for cle, dim in dimensions.items():
        orphelines = ~fact_table[cle].isin(dim[cle])
        n = int(orphelines.sum())
        print(f'{cle:<12} clés orphelines : {n}')
        if n:
            problemes[cle] = fact_table.loc[orphelines, cle].unique()[:5].tolist()

    if problemes:
        raise ValueError(f'Intégrité référentielle violée (exemples) : {problemes}')
    print('Aucune clé orpheline : intégrité OK')


check_referential_integrity(fact_sales, {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store,
})


cle_date     clés orphelines : 0
id_client    clés orphelines : 0
id_produit   clés orphelines : 0
id_magasin   clés orphelines : 0
Aucune clé orpheline : intégrité OK


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.